# 2262 step 5: do the features explain the rolling average's residual?

For each candidate feature (time open, size band, time since dormant, CQC rating) this charts the **residual**: the share minus the
pipeline's rolling average, in percentage points, one line per status, with the mean, row count and standard error per bin.
Known shares and imputed shares are shown side by side. A feature whose lines sit away from zero (beyond their standard error)
carries information the rolling average misses; flat lines near zero mean it is unlikely to help a model.

**Soft gate:** stop and ask only if every feature is flat in every well-populated bin.

**Before running:** `01_dataset_and_baselines.ipynb` has written the modelling dataset. This notebook only reads it and writes
`results/residual_by_feature.parquet`.

**Stop the notebook instance when you finish** - SageMaker bills while it is running.

## Parameters

Same as notebook 1, plus:

- `TIME_OPEN_EDGES_MONTHS` - upper edges (months, exclusive) of the time-open bands.
- `MIN_BIN_ROWS` - bins with fewer rows are not drawn or used for the gate (too noisy).
- `MIN_EFFECT_PP`, `Z_VALUE` - a bin is non-flat when its mean residual is beyond `MIN_EFFECT_PP` and beyond `Z_VALUE` standard errors.

In [ ]:
REPO_DIR = "/home/ec2-user/SageMaker/DataEngineering"  # clone made by sagemaker_setup.ipynb, on main
DATA_BUCKET = "sfc-2262-employment-datasets"
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/empstat-model-compare/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
LOCAL = False  # True when RESULTS_PREFIX is a local folder

SHARES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=02_employment_status_03_impute/"
ESTIMATES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=01_filled_posts_05_estimated/"
RATINGS_PATH = f"s3://{DATA_BUCKET}/domain=02_sfc/dataset=sfc_cqc_ratings_for_data_requests/"
MODELLING_DATA_PATH = f"{RESULTS_PREFIX}modelling_dataset.parquet"
HIDDEN_TREND_PATH = f"{RESULTS_PREFIX}hidden_trendline.parquet"
HIDDEN_DATA_PATH = f"{RESULTS_PREFIX}hidden_dataset.parquet"
RESULTS_DIR = f"{RESULTS_PREFIX}results/"

N_FOLDS = 5
FOLD_SEED = 42
SAMPLE_PERCENT = 10  # workplaces used for jumpiness and the hidden-months test
SAMPLE_SEED = 42
HIDDEN_MONTHS = 12
ROLLING_AVERAGE_PERIOD = "6mo"
EXTRAPOLATION_PERIOD = "2y"
INTERPOLATION_CAP_PERIOD = "5y"
FY_START_MONTH = 4  # headline years are financial years starting this month

# Column names are defined once, here.
LOCATION = "locationId"
DATE = "cqc_location_import_date"
SERVICE = "primary_service_type"
REGION = "current_Region"
CSSR = "current_CSSR"
ROLE = "published_job_role_label"
WORKERS = "estimate_filled_posts_by_job_role"
POSTS = "estimate_filled_posts"
TIME_REGISTERED = "time_registered"  # months
DORMANCY = "dormancy"
TIME_SINCE_DORMANT = "time_since_dormant"
RATING_DATE = "rating_date"
OVERALL_RATING = "Overall_rating"
STATUSES = ["permanent", "temporary", "bank_or_pool", "agency", "other"]
KNOWN = [f"ascwds_emp_stat_{status}_pct" for status in STATUSES]
TREND = [f"{column}_imputed_trendline" for column in KNOWN]
ROLLING = [f"{column}_rolling_avg" for column in KNOWN]
IMPUTED = [f"{column}_imputed" for column in KNOWN]
TRUTH = [f"{column}_truth" for column in KNOWN]
FOLD_SAFE = [f"{column}_fold_safe" for column in ROLLING]
HIDDEN = "hidden"
FOLD = "fold"
CELL_KEYS = [SERVICE, REGION, ROLE]

# Features built here.
SIZE_BAND = "size_band"
DORMANCY_LEVEL = "dormancy_level"
RATING = "cqc_rating"
DORMANCY_RECORDED = "dormancy_recorded"
SIZE_BAND_EDGES = [(10, "<10"), (50, "10-49"), (150, "50-149")]  # upper edges, exclusive
SIZE_BAND_TOP = "150+"
SIZE_BAND_UNKNOWN = "unknown"
SIZE_BANDS = [label for _, label in SIZE_BAND_EDGES] + [SIZE_BAND_TOP, SIZE_BAND_UNKNOWN]
DORMANT_VALUE = "Y"
DORMANCY_NOT_RECORDED = "not recorded"
DORMANCY_NONE_YET = "none yet"
DORMANT_NOW = "dormant now"
DORMANT_RECENT = "dormant 1-12m ago"
DORMANT_LONG = "dormant 13m+ ago"
DORMANT_RECENT_MONTHS = 12
DORMANCY_LEVELS = [DORMANCY_NOT_RECORDED, DORMANCY_NONE_YET, DORMANT_NOW, DORMANT_RECENT, DORMANT_LONG]
RATINGS_BEST_FIRST = ["Outstanding", "Good", "Requires improvement", "Inadequate"]
UNRATED = "unrated"
RATING_LEVELS = RATINGS_BEST_FIRST + [UNRATED]

# Result column and metric names.
VARIANT, TEST, LEVEL, GROUP, METRIC, VALUE = "variant", "test", "level", "group", "metric", "value"
HELD_OUT, GUARD = "held_out", "hidden_months"
ROWS, MAE_PP, BRIER, TOP1 = "number_of_rows", "mae_pp", "brier_score", "top_1_accuracy"
MAE_SKILL, BRIER_SKILL = "mae_skill_score", "brier_skill_score"
BASELINE_MAE, BASELINE_BRIER = "baseline_mae", "baseline_brier_score"
TREND_MAE_PP, MEAN_CHANGE = "trend_mae_pp", "mean_period_to_period_change"
CONTRIBUTORS, WINDOW_CONTRIBUTORS, YEAR = "contributing_workplaces", "window_contributing_workplaces", "financial_year"

TIME_OPEN_EDGES_MONTHS = [12, 24, 48, 96]
TIME_OPEN_BAND = "time_open_band"
MIN_BIN_ROWS = 200
MIN_EFFECT_PP = 0.5
Z_VALUE = 2.0

## Setup

In [ ]:
import sys

import polars as pl

sys.path.insert(0, REPO_DIR)
import projects._03_independent_cqc._02_employment_status.fargate.utils.impute_utils as imputeUtils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

assert imputeUtils.PCT_COLUMNS == KNOWN and imputeUtils.IMPUTED_TRENDLINE_PCT_COLUMNS == TREND
assert imputeUtils.ROLLING_AVG_PCT_COLUMNS == ROLLING and imputeUtils.IMPUTED_PCT_COLUMNS == IMPUTED

import matplotlib.pyplot as plt

## Input and output helpers

In [ ]:
import io
from pathlib import Path


def scan(path: str) -> pl.LazyFrame:
    return pl.scan_parquet(path, storage_options=STORAGE_OPTIONS)


def sink(lf: pl.LazyFrame, path: str) -> None:
    lf.sink_parquet(path, storage_options=STORAGE_OPTIONS)


def write_result(df: pl.DataFrame, name: str) -> None:
    """Write a small result as parquet under RESULTS_DIR."""
    if LOCAL:
        path = Path(RESULTS_DIR) / name
        path.parent.mkdir(parents=True, exist_ok=True)
        df.write_parquet(path)
        return
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    boto3.client("s3").put_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}", Body=buffer.getvalue())


def read_result(name: str) -> pl.DataFrame:
    """Read a result written by `write_result`."""
    if LOCAL:
        return pl.read_parquet(Path(RESULTS_DIR) / name)
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    body = boto3.client("s3").get_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}")["Body"].read()
    return pl.read_parquet(io.BytesIO(body))


in_sample = pl.col(LOCATION).hash(SAMPLE_SEED) % 100 < SAMPLE_PERCENT

## Load and band time open

Scans the modelling dataset lazily and adds the time-open band. A null time registered is "unknown".

In [ ]:
def add_time_open_band(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Band time registered (months) by `TIME_OPEN_EDGES_MONTHS`; the last band is open-ended."""
    labels = [f"<{edge // 12}y" if i == 0 else f"{TIME_OPEN_EDGES_MONTHS[i - 1] // 12}-{edge // 12}y" for i, edge in enumerate(TIME_OPEN_EDGES_MONTHS)]
    labels.append(f"{TIME_OPEN_EDGES_MONTHS[-1] // 12}y+")
    band = pl.when(pl.col(TIME_REGISTERED).is_null()).then(pl.lit("unknown"))
    for edge, label in zip(TIME_OPEN_EDGES_MONTHS, labels):
        band = band.when(pl.col(TIME_REGISTERED) < edge).then(pl.lit(label))
    return lf.with_columns(band.otherwise(pl.lit(labels[-1])).cast(pl.Enum([*labels, "unknown"])).alias(TIME_OPEN_BAND))


FEATURES = {
    "time open": TIME_OPEN_BAND,
    "size band": SIZE_BAND,
    "time since dormant": DORMANCY_LEVEL,
    "CQC rating": RATING,
}
modelling_lf = add_time_open_band(scan(MODELLING_DATA_PATH))

## Residual per bin

Per feature and source (known or imputed shares): the mean residual in pp, the rows and the standard error (`std / sqrt(rows)`),
per status and bin. Only rows with a share and a rolling average are used. Each feature is one lazy `group_by`, collected small.

In [ ]:
def residual_table(lf: pl.LazyFrame, feature: str, source: str, share_columns: list[str]) -> pl.DataFrame:
    residuals = [(pl.col(share).cast(pl.Float64) - pl.col(rolling)) * 100 for share, rolling in zip(share_columns, ROLLING)]
    summary_df = (
        lf.filter(pl.col(share_columns[0]).is_not_null() & pl.col(ROLLING[0]).is_not_null())
        .group_by(feature)
        .agg(
            pl.len().alias(ROWS),
            *[r.mean().alias(f"{status}__mean") for status, r in zip(STATUSES, residuals)],
            *[r.std().alias(f"{status}__std") for status, r in zip(STATUSES, residuals)],
        )
        .collect(engine="streaming")
    )
    return pl.concat(
        [
            summary_df.select(
                pl.lit(feature).alias("feature"),
                pl.lit(source).alias("source"),
                pl.col(feature).cast(pl.String).alias("bin"),
                pl.col(feature).to_physical().alias("order"),
                pl.lit(status).alias("status"),
                ROWS,
                pl.col(f"{status}__mean").alias("mean_pp"),
                (pl.col(f"{status}__std") / pl.col(ROWS).sqrt()).alias("standard_error_pp"),
            )
            for status in STATUSES
        ]
    )


residual_df = pl.concat(
    [
        residual_table(modelling_lf, feature, source, share_columns)
        for feature in FEATURES.values()
        for source, share_columns in [("known", KNOWN), ("imputed", IMPUTED)]
    ]
)
write_result(residual_df, "residual_by_feature.parquet")
print(residual_df.height, "rows")

## Charts

One row per feature, known shares on the left and imputed shares on the right. Each line is a status; bars are 1.96 standard errors.
Bins under `MIN_BIN_ROWS` are left out. Read for lines away from zero and for lines that differ between bins; a line that moves
with the feature is information the rolling average lacks.

In [ ]:
figure, axes = plt.subplots(len(FEATURES), 2, figsize=(13, 3.2 * len(FEATURES)), squeeze=False, sharey="row")
for row, feature in enumerate(FEATURES.values()):
    for column, source in enumerate(["known", "imputed"]):
        axis = axes[row][column]
        for status in STATUSES:
            bins_df = residual_df.filter(
                (pl.col("feature") == feature) & (pl.col("source") == source) & (pl.col("status") == status) & (pl.col(ROWS) >= MIN_BIN_ROWS)
            ).sort("order")
            axis.errorbar(bins_df["bin"], bins_df["mean_pp"], yerr=1.96 * bins_df["standard_error_pp"], marker="o", capsize=3, label=status)
        axis.axhline(0, color="grey", linewidth=0.8)
        axis.set_title(f"{feature}: {source} share minus rolling average")
        axis.set_ylabel("pp")
axes[0][0].legend(fontsize=8)
figure.tight_layout()
plt.show()

## Soft gate

Counts the well-populated bins that are non-flat (mean residual beyond `MIN_EFFECT_PP` and `Z_VALUE` standard errors) per feature and source.
**If every feature has none, stop and ask before building models;** otherwise carry on.

In [ ]:
flat_df = (
    residual_df.filter(pl.col(ROWS) >= MIN_BIN_ROWS)
    .with_columns(
        ((pl.col("mean_pp").abs() > MIN_EFFECT_PP) & (pl.col("mean_pp").abs() > Z_VALUE * pl.col("standard_error_pp"))).alias("non_flat")
    )
    .group_by("feature", "source")
    .agg(pl.len().alias("status_bins"), pl.col("non_flat").sum().alias("non_flat_status_bins"))
    .sort("feature", "source")
)
print(flat_df)
print("STOP and ask: every feature is flat" if flat_df["non_flat_status_bins"].sum() == 0 else "Carry on: at least one feature moves the residual")

## What each feature shows

Written after running, one line per feature (which statuses move, in which bins, known versus imputed):

- **Time open:**
- **Size band:**
- **Time since dormant** (including "not recorded"):
- **CQC rating:**